# Elias Intelligence Research — Full Analysis Engine

Paste your GitHub token in CONFIG. Runtime → Run all.

Live dashboard: https://ramezelias1.github.io/elias-intelligence-research/
Raw data (for Claude): https://ramezelias1.github.io/elias-intelligence-research/data.html

In [ ]:
GITHUB_TOKEN  = 'PASTE_YOUR_TOKEN_HERE'
GITHUB_REPO   = 'ramezelias1/elias-intelligence-research'
GITHUB_BRANCH = 'main'
TICKERS = {'BHP':'BHP.AX','PLS':'PLS.AX','REA':'REA.AX','ZIP':'ZIP.AX','LTR':'LTR.AX','PLTR':'PLTR'}
ROLES   = {'BHP':'ASX Large-Cap','PLS':'ASX Mid-Cap','REA':'ASX Mid-Cap','ZIP':'ASX Small-Cap','LTR':'ASX Event Stock','PLTR':'US Comparison'}
START, END    = '2024-01-01', '2025-12-31'
MOVE_THRESH, MOVE_WINDOW, PRE_WINDOW, VOL_BASE = 0.15, 30, 15, 20
WEEKLY_SWING_MIN = 3
CS_THRESHOLD  = 55
print('Config ready.')

In [ ]:
import subprocess
subprocess.run(['pip','install','yfinance','pandas','numpy','scipy','-q'], capture_output=True)
import yfinance as yf
import pandas as pd
import numpy as np
import json, base64, requests
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')
print('Libraries ready.')

In [ ]:
daily_data, weekly_data = {}, {}
for label, ticker in TICKERS.items():
    print(f'Fetching {label}...', end=' ', flush=True)
    try:
        df = yf.download(ticker, start=START, end=END, progress=False, auto_adjust=True)
        if df.empty: print('SKIP'); continue
        if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
        df = df[['Open','High','Low','Close','Volume']].dropna()
        df['vol_avg20'] = df['Volume'].rolling(20).mean()
        df['vol_ratio'] = df['Volume'] / df['vol_avg20']
        daily_data[label] = df
        wdf = yf.download(ticker, start='2022-01-01', end=END, progress=False, auto_adjust=True, interval='1wk')
        if isinstance(wdf.columns, pd.MultiIndex): wdf.columns = wdf.columns.get_level_values(0)
        weekly_data[label] = wdf[['Open','High','Low','Close','Volume']].dropna()
        print(f'OK daily={len(df)} weekly={len(weekly_data[label])}')
    except Exception as e:
        print(f'ERROR {e}')
print(f'Fetched {len(daily_data)}/6 stocks.')

In [ ]:
def find_swing_points(wdf, lookback=3):
    highs, lows, dates = wdf['High'].values, wdf['Low'].values, wdf.index
    sh, sl = [], []
    for i in range(lookback, len(highs)-lookback):
        wh = highs[i-lookback:i+lookback+1]
        if highs[i] == max(wh) and list(wh).count(highs[i]) == 1: sh.append((dates[i], highs[i]))
        wl = lows[i-lookback:i+lookback+1]
        if lows[i] == min(wl) and list(wl).count(lows[i]) == 1: sl.append((dates[i], lows[i]))
    return sh, sl

def determine_trend_at_date(wdf, target_date, min_swings=WEEKLY_SWING_MIN):
    past = wdf[wdf.index <= pd.Timestamp(target_date)].tail(52)
    if len(past) < 10: return 'neutral', 0, 0, 0, 0
    sh, sl = find_swing_points(past)
    if len(sh) < 2 or len(sl) < 2: return 'neutral', 0, 0, 0, 0
    shv = [v for _, v in sh[-8:]]
    slv = [v for _, v in sl[-8:]]
    hh = sum(1 for i in range(1,len(shv)) if shv[i] > shv[i-1])
    hl = sum(1 for i in range(1,len(slv)) if slv[i] > slv[i-1])
    lh = sum(1 for i in range(1,len(shv)) if shv[i] < shv[i-1])
    ll = sum(1 for i in range(1,len(slv)) if slv[i] < slv[i-1])
    if hh >= min_swings and hl >= min_swings: return 'uptrend', hh, hl, lh, ll
    elif lh >= min_swings and ll >= min_swings: return 'downtrend', hh, hl, lh, ll
    return 'neutral', hh, hl, lh, ll

trend_states = {}
for label, wdf in weekly_data.items():
    state, hh, hl, lh, ll = determine_trend_at_date(wdf, END)
    trend_states[label] = {'state':state,'hh_count':hh,'hl_count':hl,'lh_count':lh,'ll_count':ll,
        'description':f'Last 8 weekly swings: {hh} HH, {hl} HL, {lh} LH, {ll} LL'}
    print(f'{label}: {state.upper():12} HH={hh} HL={hl} LH={lh} LL={ll}')
print('Trend detection complete.')

In [ ]:
def detect_moves(df):
    moves, used = [], set()
    closes, dates = df['Close'], df.index.tolist()
    for i in range(len(dates)-3):
        if i in used: continue
        sp = float(closes.iloc[i])
        cutoff = dates[i] + timedelta(days=MOVE_WINDOW)
        for j in range(i+1, len(dates)):
            if dates[j] > cutoff: break
            ep = float(closes.iloc[j])
            pct = (ep - sp) / sp
            if abs(pct) >= MOVE_THRESH:
                moves.append({'start_idx':i,'end_idx':j,'start_date':str(dates[i].date()),
                    'end_date':str(dates[j].date()),'start_price':round(sp,4),'end_price':round(ep,4),
                    'pct_move':round(pct*100,2),'direction':'LONG' if pct>0 else 'SHORT','trading_days':j-i})
                used.add(i); break
    if not moves: return []
    filtered, last = [], -10
    for m in sorted(moves, key=lambda x: x['start_idx']):
        if m['start_idx']-last > 5: filtered.append(m); last=m['start_idx']
        elif abs(m['pct_move']) > abs(filtered[-1]['pct_move']): filtered[-1]=m
    return filtered

def describe_candle(row):
    body = abs(float(row['Close'])-float(row['Open']))
    total = float(row['High'])-float(row['Low'])
    if total == 0: return 'no range'
    bp = body/total
    d = 'up' if float(row['Close'])>=float(row['Open']) else 'down'
    uw = (float(row['High'])-max(float(row['Open']),float(row['Close'])))/total
    lw = (min(float(row['Open']),float(row['Close']))-float(row['Low']))/total
    parts = []
    if bp>0.7: parts.append(f'strong {d} body {bp:.0%}')
    elif bp>0.4: parts.append(f'moderate {d} body {bp:.0%}')
    else: parts.append(f'indecision {bp:.0%}')
    if uw>0.35: parts.append(f'upper wick {uw:.0%}')
    if lw>0.35: parts.append(f'lower wick {lw:.0%}')
    parts.append(f'range {(total/max(float(row["Open"]),0.001))*100:.1f}pct')
    return ' | '.join(parts)

def describe_vol(vr):
    if vr is None or (isinstance(vr,float) and np.isnan(vr)): return 'no baseline'
    if vr>=4: return f'{vr:.1f}x extreme'
    if vr>=2.5: return f'{vr:.1f}x major surge'
    if vr>=1.5: return f'{vr:.1f}x elevated'
    if vr>=0.8: return f'{vr:.1f}x normal'
    return f'{vr:.1f}x drying up'

def compression_score(window_df):
    if window_df.empty or len(window_df) < 5: return 0
    candles = [describe_candle(row) for _, row in window_df.iterrows()]
    n = len(candles)
    c1 = sum(1 for c in candles if 'indecision' in c) / n * 35
    uw = sum(1 for c in candles if 'upper wick' in c) / n
    lw = sum(1 for c in candles if 'lower wick' in c) / n
    c2 = min(uw, lw) * 2 * 30
    vols = window_df['vol_ratio'].dropna().tolist()
    c3 = (sum(1 for v in vols if 0.7<=v<=1.3)/len(vols)*20) if vols else 10
    ranges = [(float(r['High'])-float(r['Low']))/max(float(r['Close']),0.001) for _,r in window_df.iterrows()]
    if len(ranges)>=6:
        er, lr2 = np.mean(ranges[:5]), np.mean(ranges[-5:])
        c4 = min(max(0,(er-lr2)/(er+1e-9))*2,1)*15
    else: c4=7
    return round(min(c1+c2+c3+c4,100),1)

def summarise(pre_df):
    if pre_df.empty: return 'No pre-move data.'
    vols = pre_df['vol_ratio'].dropna().tolist()
    if not vols: return 'No volume baseline.'
    n = len(pre_df)
    early = np.mean(vols[:5]) if len(vols)>=5 else np.mean(vols)
    late  = np.mean(vols[-5:]) if len(vols)>=5 else np.mean(vols)
    trend = 'RISING' if late>early*1.2 else 'FALLING' if late<early*0.8 else 'STABLE'
    drift = ((float(pre_df['Close'].iloc[-1])-float(pre_df['Close'].iloc[0]))/float(pre_df['Close'].iloc[0]))*100
    hv = int((pre_df['vol_ratio']>=1.5).sum())
    descs = [describe_candle(r) for _,r in pre_df.iterrows()]
    parts = [f'Drift: {drift:+.1f}% over {n} days.',
             f'Volume: {trend} (early {early:.1f}x to late {late:.1f}x).',
             f'Elevated vol days: {hv}/{n}.']
    su=sum(1 for d in descs if 'strong up' in d)
    sd=sum(1 for d in descs if 'strong down' in d)
    ind=sum(1 for d in descs if 'indecision' in d)
    ur=sum(1 for d in descs if 'upper wick' in d)
    lr=sum(1 for d in descs if 'lower wick' in d)
    if su: parts.append(f'Strong up candles: {su}.')
    if sd: parts.append(f'Strong down candles: {sd}.')
    if ind: parts.append(f'Indecision candles: {ind}.')
    if ur: parts.append(f'Upper wick rejections: {ur}.')
    if lr: parts.append(f'Lower wick rejections: {lr}.')
    return ' '.join(parts)

print('Functions ready.')

In [ ]:
stocks_results = {}
for label, df in daily_data.items():
    wdf = weekly_data.get(label)
    moves = detect_moves(df)
    print(f'{label}: {len(moves)} moves')
    analysed = []
    for move in moves:
        pre_df   = df.iloc[max(0,move['start_idx']-PRE_WINDOW):move['start_idx']].copy()
        trig_row = df.iloc[move['start_idx']]
        cs = compression_score(pre_df)
        t_state, hh, hl, lh, ll = determine_trend_at_date(wdf, move['start_date']) if wdf is not None else ('neutral',0,0,0,0)
        pre_days = []
        for di,(date,row) in enumerate(pre_df.iterrows()):
            vr = float(row['vol_ratio']) if not pd.isna(row['vol_ratio']) else None
            pre_days.append({'day_offset':di-len(pre_df)+1,'date':str(date.date()),
                'open':round(float(row['Open']),4),'high':round(float(row['High']),4),
                'low':round(float(row['Low']),4),'close':round(float(row['Close']),4),
                'volume':int(row['Volume']),'vol_ratio':round(vr,2) if vr else None,
                'candle_desc':describe_candle(row),'volume_desc':describe_vol(vr)})
        tvr = float(trig_row['vol_ratio']) if not pd.isna(trig_row['vol_ratio']) else None
        trig = {'date':str(df.index[move['start_idx']].date()),
            'open':round(float(trig_row['Open']),4),'high':round(float(trig_row['High']),4),
            'low':round(float(trig_row['Low']),4),'close':round(float(trig_row['Close']),4),
            'volume':int(trig_row['Volume']),'vol_ratio':round(tvr,2) if tvr else None,
            'candle_desc':describe_candle(trig_row),'volume_desc':describe_vol(tvr)}
        analysed.append({'move':move,'trigger_day':trig,'pre_days':pre_days,
            'summary':summarise(pre_df),'compression_score':cs,
            'trend_state':t_state,'trend_hh':hh,'trend_hl':hl,'trend_lh':lh,'trend_ll':ll})
    stocks_results[label] = {'label':label,'ticker':TICKERS[label],'role':ROLES[label],
        'total_days':len(df),'date_range':f"{df.index[0].date()} to {df.index[-1].date()}",
        'moves':analysed}
print('Move analysis complete.')

In [ ]:
backtest_signals = []
for label, df in daily_data.items():
    wdf = weekly_data.get(label)
    dates, closes = df.index.tolist(), df['Close']
    print(f'Backtesting {label}...', end=' ', flush=True)
    n_sig = 0
    for i in range(PRE_WINDOW, len(dates)-5):
        cs = compression_score(df.iloc[i-PRE_WINDOW:i])
        if cs < CS_THRESHOLD: continue
        signal_date = str(dates[i].date())
        t_state = determine_trend_at_date(wdf, signal_date)[0] if wdf is not None else 'neutral'
        sp = float(closes.iloc[i])
        cutoff = dates[i] + timedelta(days=MOVE_WINDOW)
        outcome, best_pct, direction = 'no_move', 0.0, None
        for j in range(i+1, min(i+35,len(dates))):
            if dates[j] > cutoff: break
            pct = (float(closes.iloc[j]) - sp) / sp
            if abs(pct) > abs(best_pct): best_pct = pct
            if abs(pct) >= MOVE_THRESH:
                direction = 'LONG' if pct>0 else 'SHORT'
                outcome = direction; break
        aligned = (t_state=='uptrend' and direction=='LONG') or (t_state=='downtrend' and direction=='SHORT')
        backtest_signals.append({'label':label,'signal_date':signal_date,'cs':cs,
            'trend_state':t_state,'outcome':outcome,'direction':direction,
            'best_pct':round(best_pct*100,2),'aligned':aligned})
        n_sig += 1
    print(f'{n_sig} signals')

total = len(backtest_signals)
hits  = [s for s in backtest_signals if s['outcome'] != 'no_move']
al_s  = [s for s in backtest_signals if s['trend_state'] != 'neutral']
al_h  = [s for s in al_s if s['aligned']]
sr_any = round(len(hits)/total*100,1) if total else 0
sr_al  = round(len(al_h)/len(al_s)*100,1) if al_s else 0
avg_m  = round(np.mean([abs(s['best_pct']) for s in hits]),2) if hits else 0
fr     = round((1-len(hits)/total)*100,1) if total else 0
by_stock = {}
for label in TICKERS:
    ss=[s for s in backtest_signals if s['label']==label]
    sh=[s for s in ss if s['outcome']!='no_move']
    sa=[s for s in ss if s['trend_state']!='neutral']
    sah=[s for s in sa if s['aligned']]
    by_stock[label]={'total_signals':len(ss),
        'success_rate':round(len(sh)/len(ss)*100,1) if ss else 0,
        'trend_aligned_rate':round(len(sah)/len(sa)*100,1) if sa else 0,
        'avg_move':round(np.mean([abs(s['best_pct']) for s in sh]),2) if sh else 0}
print(f'Total={total} Success={sr_any}% TrendAligned={sr_al}% AvgMove={avg_m}% FalseRate={fr}%')

In [ ]:
# ── Build data.html (plain HTML, readable by Claude fetch tool) ──────────
# and index.html (interactive dashboard for you)
# data.html has NO JavaScript — all data in plain HTML tables

gen = datetime.now().strftime('%d %b %Y %H:%M')
all_moves = [m for s in stocks_results.values() for m in s['moves']]
longs  = sum(1 for m in all_moves if m['move']['direction']=='LONG')
shorts = sum(1 for m in all_moves if m['move']['direction']=='SHORT')
avg_cs = round(np.mean([m['compression_score'] for m in all_moves]),1) if all_moves else 0

# ── BUILD data.html — pure HTML, no JS, fully readable by fetch ──────────
rows = []
rows.append(f'<html><head><meta charset="UTF-8"><title>Elias Research Data</title></head><body>')
rows.append(f'<h1>ELIAS INTELLIGENCE RESEARCH — RAW DATA</h1>')
rows.append(f'<p>Generated: {gen} | Stocks: {len(stocks_results)} | Total Moves: {len(all_moves)} | Long: {longs} | Short: {shorts} | Avg CS: {avg_cs}</p>')

# Trend states
rows.append('<h2>TREND STATES (Wealth Within)</h2>')
rows.append('<table border="1"><tr><th>Stock</th><th>State</th><th>HH</th><th>HL</th><th>LH</th><th>LL</th><th>Description</th></tr>')
for label, t in trend_states.items():
    rows.append(f'<tr><td>{label}</td><td>{t["state"]}</td><td>{t["hh_count"]}</td><td>{t["hl_count"]}</td><td>{t["lh_count"]}</td><td>{t["ll_count"]}</td><td>{t["description"]}</td></tr>')
rows.append('</table>')

# Backtest summary
rows.append('<h2>BACKTEST RESULTS</h2>')
rows.append(f'<p>Total Signals: {total} | Success Rate: {sr_any}% | Trend-Aligned Rate: {sr_al}% | Avg Move: {avg_m}% | False Rate: {fr}% | CS Threshold: {CS_THRESHOLD}</p>')
rows.append('<table border="1"><tr><th>Stock</th><th>Signals</th><th>Success%</th><th>TrendAligned%</th><th>AvgMove%</th></tr>')
for label, s in by_stock.items():
    rows.append(f'<tr><td>{label}</td><td>{s["total_signals"]}</td><td>{s["success_rate"]}</td><td>{s["trend_aligned_rate"]}</td><td>{s["avg_move"]}</td></tr>')
rows.append('</table>')

# All moves with pre-move data
rows.append('<h2>ALL QUALIFYING MOVES</h2>')
rows.append('<table border="1"><tr><th>Stock</th><th>Role</th><th>Direction</th><th>Start</th><th>End</th><th>Start$</th><th>End$</th><th>Pct%</th><th>Days</th><th>TrendState</th><th>CS</th><th>Summary</th></tr>')
for label, stock in stocks_results.items():
    for am in stock['moves']:
        m = am['move']
        rows.append(f'<tr><td>{label}</td><td>{stock["role"]}</td><td>{m["direction"]}</td><td>{m["start_date"]}</td><td>{m["end_date"]}</td><td>{m["start_price"]}</td><td>{m["end_price"]}</td><td>{m["pct_move"]}</td><td>{m["trading_days"]}</td><td>{am["trend_state"]}</td><td>{am["compression_score"]}</td><td>{am["summary"]}</td></tr>')
        # Pre-move days
        rows.append('<tr><td colspan="12"><table border="1" style="margin:4px"><tr><th>Day</th><th>Date</th><th>Close</th><th>VolRatio</th><th>VolDesc</th><th>CandleDesc</th></tr>')
        for d in am['pre_days']:
            rows.append(f'<tr><td>{d["day_offset"]}</td><td>{d["date"]}</td><td>{d["close"]}</td><td>{d["vol_ratio"]}</td><td>{d["volume_desc"]}</td><td>{d["candle_desc"]}</td></tr>')
        td = am['trigger_day']
        rows.append(f'<tr style="background:#ffe"><td>TRIGGER</td><td>{td["date"]}</td><td>{td["close"]}</td><td>{td["vol_ratio"]}</td><td>{td["volume_desc"]}</td><td>{td["candle_desc"]}</td></tr>')
        rows.append('</table></td></tr>')
rows.append('</table>')
rows.append('</body></html>')

data_html = '\n'.join(rows)
print(f'data.html size: {len(data_html):,} chars')

# ── Push to GitHub ────────────────────────────────────────────────────────
def push_to_github(filename, content, token, repo, branch='main'):
    url = f'https://api.github.com/repos/{repo}/contents/{filename}'
    hdrs = {'Authorization': f'token {token}', 'Accept': 'application/vnd.github.v3+json'}
    r = requests.get(url, headers=hdrs)
    sha = r.json().get('sha') if r.status_code==200 else None
    payload = {'message': f'Update {filename} — {gen}',
               'content': base64.b64encode(content.encode()).decode(), 'branch': branch}
    if sha: payload['sha'] = sha
    r = requests.put(url, headers=hdrs, json=payload)
    status = 'OK' if r.status_code in (200,201) else f'ERROR {r.status_code}: {r.text[:200]}'
    print(f'  {filename}: {status}')
    return r.status_code in (200,201)

if GITHUB_TOKEN == 'PASTE_YOUR_TOKEN_HERE':
    print('No token — skipping push.')
else:
    print('Pushing to GitHub...')
    push_to_github('data.html', data_html, GITHUB_TOKEN, GITHUB_REPO)
    print('Done!')
    print('Claude can now fetch: https://ramezelias1.github.io/elias-intelligence-research/data.html')